In [1]:
from shapely.geometry import box, Point, LineString, Polygon, MultiPolygon
from shapely.ops import unary_union
import geopandas as gpd
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling
import osmnx as ox  
import contextily as ctx
from geopy.geocoders import Nominatim  
from geopy.extra.rate_limiter import RateLimiter
import os
#!pip install h3
import h3
import matplotlib.patches as mpatches

In [2]:
# Get the current working directory
current_path = os.getcwd()

# Define relative paths for data, output, and helper functions
data_in = os.path.join(current_path, "data")       # Folder with input data
output = os.path.join(current_path, "output")      # Folder for saving results

## Green spaces

In [3]:
file_name = 'data/OEFFGRUENFLOGD.json'
gdf_green = gpd.read_file(file_name)
print(gdf_green.columns)
print(len(gdf_green))
print(gdf_green['F_TYP'].value_counts())
gdf_green.head(3)

Index(['id', 'OBJECTID', 'TEXT13ID', 'F_TYP', 'T_TEXT', 'T_LANG', 'O', 'P',
       'O_UEREG', 'O_REGION', 'O_LOKAL', 'P_WLD', 'P_WAS', 'P_WIE', 'P_WGT',
       'P_LW', 'P_PK', 'P_GPK', 'P_GST', 'P_SGR', 'P_AGR', 'P_RGR', 'P_URB',
       'Q_MOB', 'Q_SPI', 'Q_GRI', 'Q_HUN', 'Q_BAB', 'Q_WC', 'Q_SPO', 'Q_TRB',
       'UMFANG', 'FLAECHE', 'SE_ANNO_CAD_DATA', 'geometry'],
      dtype='object')
1936
F_TYP
L    1629
U     171
R     136
Name: count, dtype: int64


,id,OBJECTID,TEXT13ID,F_TYP,T_TEXT,T_LANG,O,P,O_UEREG,O_REGION,...,Q_GRI,Q_HUN,Q_BAB,Q_WC,Q_SPO,Q_TRB,UMFANG,FLAECHE,SE_ANNO_CAD_DATA,geometry
0,OEFFGRUENFLOGD.2289,2289,Währinger Pa*,L,Währinger Park,Währinger Park,1,100000,0,0,...,0,1,0,1,1245,1,1460.941451,57020.131903,None,"MULTIPOLYGON (((16.35083 48.23369, 16.34929 48..."
1,OEFFGRUENFLOGD.2290,2290,Türkenschanz*,L,Türkenschanzplatz,Türkenschanzplatz,1,1,0,0,...,0,0,0,0,0,0,310.900196,2002.715993,None,"POLYGON ((16.32886 48.23482, 16.32824 48.23492..."
2,OEFFGRUENFLOGD.2291,2291,Kongreßpark,L,Kongreßpark/Kainzga,Kainzgasse,1,100000,0,0,...,0,1,0,2,12,1,249.284876,3243.484789,None,"POLYGON ((16.30871 48.22281, 16.30933 48.22285..."


In [4]:
print(gdf_green.crs)
gdf_green = gdf_green[['id', 'F_TYP', 'T_TEXT', 'FLAECHE', 'geometry']]
gdf_green = gdf_green.rename(columns={'FLAECHE': 'original_greenarea',
                                     'geometry': 'geometry_green'})
gdf_green.set_geometry('geometry_green', inplace=True)

gdf_green = gdf_green.to_crs(epsg=31256)
gdf_green['area_m2'] = gdf_green.geometry.area
print(gdf_green.crs)
gdf_green.head(3)

EPSG:4326
EPSG:31256


,id,F_TYP,T_TEXT,original_greenarea,geometry_green,area_m2
0,OEFFGRUENFLOGD.2289,L,Währinger Park,57020.131903,"MULTIPOLYGON (((1388.829 343923.309, 1274.719 ...",57020.220876
1,OEFFGRUENFLOGD.2290,L,Türkenschanzplatz,2002.715993,"POLYGON ((-242.971 344049.152, -288.821 344059...",2002.715999
2,OEFFGRUENFLOGD.2291,L,Kongreßpark/Kainzga,3243.484789,"POLYGON ((-1740.791 342714.09, -1694.181 34271...",3243.484798


In [5]:
print(len(gdf_green))
print("Sum green areas - from Flaeche:", gdf_green["original_greenarea"].sum())
print("Sum green areas - my calculation:", gdf_green['area_m2'].sum())

1936
Sum green areas - from Flaeche: 174781716.3472637
Sum green areas - my calculation: 177485054.0345264


## Census districts

In [6]:
boundaries_path = 'data/ZAEHLBEZIRKOGDPolygon.shp'
census_boundaries = gpd.read_file(boundaries_path)
print(census_boundaries.crs)
census_boundaries = census_boundaries.to_crs(epsg=31256)
census_boundaries.head(3)

EPSG:4326


,BEZNR,ZBEZNR,BEZ,ZBEZ,FLAECHE,UMFANG,AKT_TIMEST,SE_SDO_ROW,geometry
0,22.0,19.0,22,2219,431562.4878,2963.93,2025-05-10,305665.0,"POLYGON ((9534.781 343188.416, 9466.411 343046..."
1,19.0,9.0,19,1909,700939.3051,3670.56,2025-05-10,305666.0,"POLYGON ((419.711 344179.176, 419.451 344291.5..."
2,9.0,2.0,09,0902,471286.9574,3510.43,2025-05-10,305667.0,"POLYGON ((2280.931 343474.716, 2281.161 343474..."


In [7]:
census_boundaries = census_boundaries[['ZBEZ', 'FLAECHE', 'geometry']]
census_boundaries = census_boundaries.rename(columns={'FLAECHE': 'original_censusarea',
                                                     'geometry': 'geometry_census'})
census_boundaries.set_geometry('geometry_census', inplace=True)
census_boundaries['area_boundaries'] = census_boundaries.geometry.area
census_boundaries.head(3)

,ZBEZ,original_censusarea,geometry_census,area_boundaries
0,2219,431562.4878,"POLYGON ((9534.781 343188.416, 9466.411 343046...",431562.489038
1,1909,700939.3051,"POLYGON ((419.711 344179.176, 419.451 344291.5...",700939.307083
2,0902,471286.9574,"POLYGON ((2280.931 343474.716, 2281.161 343474...",471286.958700


In [8]:
print(len(census_boundaries))
print("Sum census district areas - from Flaeche:", census_boundaries["original_censusarea"].sum())
print("Sum census district areas - my calculation:", census_boundaries['area_boundaries'].sum())

250
Sum census district areas - from Flaeche: 414870565.6894
Sum census district areas - my calculation: 414870566.8331835


In [9]:
print(census_boundaries.crs)
print(gdf_green.crs)

EPSG:31256
EPSG:31256


In [10]:
# green area per census district
green_in_census = gpd.overlay(census_boundaries, gdf_green, how='intersection')
# green_in_census = gpd.overlay(census_boundaries, gdf_green, how='intersection', keep_geom_type=False)
print(len(green_in_census))
print(green_in_census.isnull().sum())
green_in_census['new_green_area'] = green_in_census.geometry.area
print("Original green area:", gdf_green['area_m2'].sum())
print("Green area after overlay:", green_in_census['new_green_area'].sum())
green_in_census.head(3)

2414
ZBEZ                   0
original_censusarea    0
area_boundaries        0
id                     0
F_TYP                  0
T_TEXT                 0
original_greenarea     0
area_m2                0
geometry               0
dtype: int64
Original green area: 177485054.0345264
Green area after overlay: 126944693.16638353


,ZBEZ,original_censusarea,area_boundaries,id,F_TYP,T_TEXT,original_greenarea,area_m2,geometry,new_green_area
0,2219,431562.4878,431562.489038,OEFFGRUENFLOGD.2676,L,Mühlgrund,76517.700862,76517.701080,"MULTIPOLYGON (((9103.751 342516.125, 9116.158 ...",247.320048
1,2219,431562.4878,431562.489038,OEFFGRUENFLOGD.2677,L,Hausgrun dweg,2739.814240,2739.814248,"POLYGON ((9466.411 343046.736, 9466.404 343046...",2737.393493
2,2219,431562.4878,431562.489038,OEFFGRUENFLOGD.2678,L,Vernholzgasse,5103.515597,5103.515612,"POLYGON ((8986.451 343400.036, 8995.931 343420...",158.497458


In [11]:
green_in_census = green_in_census.sort_values(by=['ZBEZ', 'id']).reset_index(drop=True)
green_in_census.head(5)

# grouping green areas based on their id to check whether same green area is divided into multiple census districts
green_in_census.groupby('id').size().reset_index(name='count').sort_values(by='count', ascending=False).head(20)
print("Sum green areas per census district - after my groupby:", green_in_census["new_green_area"].sum())

#print(green_in_census[green_in_census['id']== 'OEFFGRUENFLOGD.3713']['new_area'].sum())
#green_in_census[green_in_census['id']== 'OEFFGRUENFLOGD.3713']
print(len(green_in_census))
print(green_in_census['ZBEZ'].nunique())
green_in_census.head(3)

Sum green areas per census district - after my groupby: 126944693.16638353
2414
248


,ZBEZ,original_censusarea,area_boundaries,id,F_TYP,T_TEXT,original_greenarea,area_m2,geometry,new_green_area
0,0101,377461.6586,377461.659643,OEFFGRUENFLOGD.2838,L,Dr.-Karl-Lueger-Platz,1804.722495,1804.722500,"POLYGON ((3496.99 341023.561, 3494.089 341024....",37.070825
1,0101,377461.6586,377461.659643,OEFFGRUENFLOGD.3308,R,DonaukanalFranzJosefskai,7193.400254,7193.400274,"POLYGON ((3486.181 341497.856, 3487.921 341497...",1139.957538
2,0102,379376.7503,379376.751348,OEFFGRUENFLOGD.2347,R,Stadtpark,108386.138482,108386.138793,"MULTIPOLYGON (((3723.931 340853.976, 3721.311 ...",69630.066640


In [12]:
## identifying lost green areas (50million m2)
# green areas that has no intersect with census district boundaries at all
lost_green_ids = set(gdf_green['id']) - set(green_in_census['id'])
print(f"Number of green areas that is not in any census district: {len(lost_green_ids)}")
print(f"Green areas that is not in any census district: {lost_green_ids}")


green_clipped_per_id = green_in_census.groupby('id')['new_green_area'].sum().reset_index()
total_full_green_area = gdf_green['area_m2'].sum()

green_area_comparison = gdf_green[['id', 'area_m2']].merge(green_clipped_per_id, on='id', how='left')
green_area_comparison['new_green_area'] = green_area_comparison['new_green_area'].fillna(0)
green_area_comparison['loss_m2'] = green_area_comparison['area_m2'] - green_area_comparison['new_green_area']
green_area_comparison['loss_pct'] = (green_area_comparison['loss_m2'] / green_area_comparison['area_m2']) * 100

Number of green areas that is not in any census district: 53
Green areas that is not in any census district: {'OEFFGRUENFLOGD.4021', 'OEFFGRUENFLOGD.3864', 'OEFFGRUENFLOGD.3919', 'OEFFGRUENFLOGD.3929', 'OEFFGRUENFLOGD.3878', 'OEFFGRUENFLOGD.3876', 'OEFFGRUENFLOGD.3993', 'OEFFGRUENFLOGD.3907', 'OEFFGRUENFLOGD.3953', 'OEFFGRUENFLOGD.3897', 'OEFFGRUENFLOGD.4010', 'OEFFGRUENFLOGD.3940', 'OEFFGRUENFLOGD.3872', 'OEFFGRUENFLOGD.3930', 'OEFFGRUENFLOGD.4059', 'OEFFGRUENFLOGD.3913', 'OEFFGRUENFLOGD.3916', 'OEFFGRUENFLOGD.3854', 'OEFFGRUENFLOGD.3871', 'OEFFGRUENFLOGD.3875', 'OEFFGRUENFLOGD.3923', 'OEFFGRUENFLOGD.3900', 'OEFFGRUENFLOGD.3860', 'OEFFGRUENFLOGD.3920', 'OEFFGRUENFLOGD.3874', 'OEFFGRUENFLOGD.3906', 'OEFFGRUENFLOGD.3912', 'OEFFGRUENFLOGD.4004', 'OEFFGRUENFLOGD.3961', 'OEFFGRUENFLOGD.3879', 'OEFFGRUENFLOGD.3870', 'OEFFGRUENFLOGD.3932', 'OEFFGRUENFLOGD.3863', 'OEFFGRUENFLOGD.3853', 'OEFFGRUENFLOGD.3918', 'OEFFGRUENFLOGD.3908', 'OEFFGRUENFLOGD.3867', 'OEFFGRUENFLOGD.4020', 'OEFFGRUENFLOGD.

In [13]:
print(len(green_area_comparison))
green_area_comparison.head()

1936


,id,area_m2,new_green_area,loss_m2,loss_pct
0,OEFFGRUENFLOGD.2289,57020.220876,57020.220876,7.785275e-10,1.365353e-12
1,OEFFGRUENFLOGD.2290,2002.715999,2002.715999,-1.271701e-09,-6.349882e-11
2,OEFFGRUENFLOGD.2291,3243.484798,3243.484798,-3.210516e-10,-9.898355e-12
3,OEFFGRUENFLOGD.2292,5086.636449,5086.636449,-3.637979e-12,-7.152032e-14
4,OEFFGRUENFLOGD.2293,4592.970413,4592.970413,0.000000e+00,0.000000e+00


In [14]:
nogreen_districts = set(census_boundaries['ZBEZ']) - set(green_in_census['ZBEZ'])
print(f"Census districts with no green area: {nogreen_districts}")
print(f"Number of green areas that is not in any census district: {len(lost_green_ids)}")

Census districts with no green area: {'2231', '2120'}
Number of green areas that is not in any census district: 53


In [15]:
green_area_per_census = green_in_census.groupby('ZBEZ')['new_green_area'].sum().reset_index()
print("Sum green areas per census district - my calculation, after groupby:", green_area_per_census["new_green_area"].sum())

# Merge with ALL census boundaries — keeps all 250
green_area_per_census = census_boundaries[['ZBEZ', 'area_boundaries', 'geometry_census']].merge(
    green_area_per_census, on='ZBEZ', how='left'
)

# fill missing values with 0 (districts with no green space)
green_area_per_census['new_green_area'] = green_area_per_census['new_green_area'].fillna(0)

# green ratio calculation
green_area_per_census['green_ratio'] = green_area_per_census['new_green_area'] / green_area_per_census['area_boundaries']

print(len(green_area_per_census))
print(green_area_per_census.isnull().sum())
green_area_per_census.head(3)

Sum green areas per census district - my calculation, after groupby: 126944693.16638355
250
ZBEZ               0
area_boundaries    0
geometry_census    0
new_green_area     0
green_ratio        0
dtype: int64


,ZBEZ,area_boundaries,geometry_census,new_green_area,green_ratio
0,2219,431562.489038,"POLYGON ((9534.781 343188.416, 9466.411 343046...",3143.210999,0.007283
1,1909,700939.307083,"POLYGON ((419.711 344179.176, 419.451 344291.5...",9485.049730,0.013532
2,0902,471286.958700,"POLYGON ((2280.931 343474.716, 2281.161 343474...",33699.286438,0.071505


In [16]:
# Number of census districts where green_ratio > 1
problem_count_g = (green_area_per_census['green_ratio'] > 1).sum()
print("Districts with green_ratio > 1:", problem_count_g)

Districts with green_ratio > 1: 0


## Including socio-economic variables

In [17]:
data_ssi = gpd.read_file('data/data_ssi.geojson')
print(data_ssi.crs)
print(len(data_ssi))

data_ssi = data_ssi.to_crs(epsg=31256)
data_ssi['area_ssi'] = data_ssi.geometry.area
# data_ssi = data_ssi[['ZBEZ', 'area_ssi']]
data_ssi.head(5)

EPSG:4326
246


,ZBEZ,university_share,avg_income,unemprate,benefit_share,SSI,quintile,geometry,area_ssi
0,0101,1.604954,1.741399,1.125923,1.123008,1.653992,5,"POLYGON ((3485.019 341498.126, 3484.459 341498...",377557.211380
1,0102,1.653812,2.565233,1.237000,1.139440,2.000434,5,"POLYGON ((3487.789 341009.066, 3470.019 341017...",379376.751386
2,0103,1.706598,3.805033,1.641029,1.232519,2.570582,5,"POLYGON ((2735.149 340257.116, 2743.919 340257...",465496.860383
3,0104,1.845975,3.156000,1.697713,1.313290,2.407460,5,"POLYGON ((2313.839 340611.086, 2356.209 340648...",835823.045243
4,0105,1.681303,2.424953,1.192502,1.252134,1.971322,5,"POLYGON ((2639.529 341630.546, 2697.199 341592...",333293.038103


In [28]:
'''
# sanity check to ensure 2 census data sets are for the exact same polygons
merged = census_boundaries[['ZBEZ', 'geometry_census']].merge(
    data_ssi[['ZBEZ', 'geometry']], on='ZBEZ', how='left'
)

merged['same_geometry'] = merged.apply(
    lambda row: row['geometry'].almost_equals(row['geometry_census']) if pd.notnull(row['geometry']) and pd.notnull(row['geometry_census']) else False,
    axis=1
)

merged['geometry_missing'] = merged['geometry'].isna()
print("Matching geometries:", merged['same_geometry'].sum())
print("Total districts compared:", len(merged))
merged.head()
'''

Matching geometries: 0
Total districts compared: 250


/var/folders/fg/tx5hq3917p5dxc674m9h_4500000gn/T/ipykernel_36400/798533831.py:7: ShapelyDeprecationWarning: The 'almost_equals()' method is deprecated and will be removed in Shapely 2.1; use 'equals_exact()' instead
  lambda row: row['geometry'].almost_equals(row['geometry_census']) if pd.notnull(row['geometry']) and pd.notnull(row['geometry_census']) else False,


,ZBEZ,geometry_census,geometry,same_geometry,geometry_missing
0,2219,"POLYGON ((9534.781 343188.416, 9466.411 343046...","POLYGON ((9548.449 343216.676, 9477.569 343253...",False,False
1,1909,"POLYGON ((419.711 344179.176, 419.451 344291.5...","POLYGON ((419.719 344179.166, 420.099 343999.9...",False,False
2,0902,"POLYGON ((2280.931 343474.716, 2281.161 343474...","POLYGON ((2280.939 343474.706, 2280.809 343475...",False,False
3,0801,"POLYGON ((1411.261 341840.466, 1562.091 341826...","POLYGON ((1411.269 341840.456, 1401.149 341841...",False,False
4,0502,"POLYGON ((2588.871 338593.076, 2649.781 338500...","POLYGON ((2588.879 338593.066, 2531.579 338680...",False,False


In [24]:
# Merge green ratios with data_ssi (socio-economic variables)
merged_census = green_area_per_census.merge(
    data_ssi, on='ZBEZ', how='left'  # preserve all districts from green_area_per_census
)

missing_ssi = merged_census[merged_census['geometry'].isna()]
print("Districts missing in data_ssi:", missing_ssi['ZBEZ'].tolist())
print(merged_census['new_green_area'].sum())
merged_census.head()

Districts missing in data_ssi: ['2230', '1310', '0210', '2130']
126944693.16638356


,ZBEZ,area_boundaries,geometry_census,new_green_area,green_ratio,university_share,avg_income,unemprate,benefit_share,SSI,quintile,geometry,area_ssi
0,2219,431562.489038,"POLYGON ((9534.781 343188.416, 9466.411 343046...",3143.210999,0.007283,-0.824343,-0.666530,-0.279051,-0.514422,-0.698326,2.0,"POLYGON ((9548.449 343216.676, 9477.569 343253...",429854.038477
1,1909,700939.307083,"POLYGON ((419.711 344179.176, 419.451 344291.5...",9485.049730,0.013532,1.861136,2.007536,1.161796,1.140429,1.857084,5.0,"POLYGON ((419.719 344179.166, 420.099 343999.9...",700939.307152
2,0902,471286.958700,"POLYGON ((2280.931 343474.716, 2281.161 343474...",33699.286438,0.071505,1.534756,0.389211,0.873002,0.970293,1.052743,5.0,"POLYGON ((2280.939 343474.706, 2280.809 343475...",471286.958747
3,0801,267754.534140,"POLYGON ((1411.261 341840.466, 1562.091 341826...",11222.721443,0.041914,1.437628,0.508873,0.996835,1.063242,1.101179,5.0,"POLYGON ((1411.269 341840.456, 1401.149 341841...",267740.400967
4,0502,671172.393908,"POLYGON ((2588.871 338593.076, 2649.781 338500...",28936.427128,0.043113,-0.179935,-0.967178,-0.977547,-0.903215,-0.772275,2.0,"POLYGON ((2588.879 338593.066, 2531.579 338680...",671018.541225


In [25]:
# Drop rows where 'income' or 'unemployed_rate' is NaN
census_final = merged_census.dropna(subset=['avg_income', 'unemprate'])
print(len(census_final))
print(census_final['new_green_area'].sum())
census_final.head(3)

246
82170714.99490605


,ZBEZ,area_boundaries,geometry_census,new_green_area,green_ratio,university_share,avg_income,unemprate,benefit_share,SSI,quintile,geometry,area_ssi
0,2219,431562.489038,"POLYGON ((9534.781 343188.416, 9466.411 343046...",3143.210999,0.007283,-0.824343,-0.666530,-0.279051,-0.514422,-0.698326,2.0,"POLYGON ((9548.449 343216.676, 9477.569 343253...",429854.038477
1,1909,700939.307083,"POLYGON ((419.711 344179.176, 419.451 344291.5...",9485.049730,0.013532,1.861136,2.007536,1.161796,1.140429,1.857084,5.0,"POLYGON ((419.719 344179.166, 420.099 343999.9...",700939.307152
2,0902,471286.958700,"POLYGON ((2280.931 343474.716, 2281.161 343474...",33699.286438,0.071505,1.534756,0.389211,0.873002,0.970293,1.052743,5.0,"POLYGON ((2280.939 343474.706, 2280.809 343475...",471286.958747
